## Example 1: Context and socket creation

In [ ]:
# Example 1: Context and socket creation
import zmq  # pyzmq is imported as 'zmq'

print(f"libzmq version: {zmq.zmq_version()}")
print(f"pyzmq version:  {zmq.pyzmq_version()}")
assert len(zmq.zmq_version_info()) == 3 and zmq.zmq_version_info()[0] >= 4, zmq.zmq_version_info()

context = zmq.Context()
socket_types = {"REQ": zmq.REQ, "REP": zmq.REP, "PUB": zmq.PUB, "SUB": zmq.SUB,
                "PUSH": zmq.PUSH, "PULL": zmq.PULL, "PAIR": zmq.PAIR,
                "DEALER": zmq.DEALER, "ROUTER": zmq.ROUTER}

sockets = {}
for type_name, socket_type in socket_types.items():
    sock = context.socket(socket_type)
    sockets[type_name] = sock
    print(f"Created {type_name:6} socket (type constant {socket_type})")
    assert sock.socket_type == socket_type
    assert sock.getsockopt(zmq.TYPE) == socket_type

for sock in sockets.values():
    sock.close(linger=0)
assert all(sock.closed for sock in sockets.values())

context.term()
assert context.closed
print(f"Closed {len(sockets)} sockets and terminated the context")

## Example 2: Request-reply over TCP on the loopback interface

In [ ]:
# Example 2: Request-reply over TCP on the loopback interface
# Every receive has a 2-second timeout, so a lost message raises zmq.Again and fails the cell.
import zmq

context = zmq.Context()
server = context.socket(zmq.REP)
client = context.socket(zmq.REQ)
for sock in (server, client):
    sock.setsockopt(zmq.LINGER, 0)
    sock.setsockopt(zmq.RCVTIMEO, 2000)

port = server.bind_to_random_port("tcp://127.0.0.1")
client.connect(f"tcp://127.0.0.1:{port}")
print(f"REP socket bound to tcp://127.0.0.1:{port}")

try:
    for i in range(1, 4):
        request = {"id": f"req_{i}", "data": f"Hello from client {i}"}
        client.send_json(request)

        received = server.recv_json()
        assert received == request, received
        server.send_json({"status": "success", "echo": received["data"], "request_id": received["id"]})

        reply = client.recv_json()
        print(f"{request['id']}: {reply}")
        assert reply == {"status": "success", "echo": request["data"], "request_id": request["id"]}, reply
finally:
    client.close()
    server.close()
    context.term()

print("3 request-reply round trips completed")

## Example 3: Publish-subscribe with topic filtering

In [ ]:
# Example 3: Publish-subscribe with topic filtering
# The publisher is an XPUB socket: it receives each subscription as a message, so the cell can wait until all
# three subscribers are subscribed before publishing (a plain PUB socket would drop early messages).
import json

import zmq

context = zmq.Context()
publisher = context.socket(zmq.XPUB)
publisher.setsockopt(zmq.LINGER, 0)
publisher.setsockopt(zmq.RCVTIMEO, 2000)
port = publisher.bind_to_random_port("tcp://127.0.0.1")

topics = ["news", "sports", "weather"]
subscribers = {}
for topic in topics:
    sub = context.socket(zmq.SUB)
    sub.setsockopt(zmq.LINGER, 0)
    sub.setsockopt(zmq.RCVTIMEO, 2000)
    sub.connect(f"tcp://127.0.0.1:{port}")
    sub.setsockopt_string(zmq.SUBSCRIBE, topic)
    subscribers[topic] = sub

try:
    # Each subscription arrives as b"\x01" + topic
    seen = {publisher.recv()[1:].decode() for _ in topics}
    assert seen == set(topics), seen

    messages = [
        ("news", {"headline": "Breaking: New discovery"}),
        ("sports", {"headline": "Championship finals tonight"}),
        ("weather", {"headline": "Sunny skies expected"}),
        ("news", {"headline": "Economy shows growth"}),
        ("sports", {"headline": "Trade deadline approaches"}),
        ("weather", {"headline": "Rain forecast tomorrow"}),
    ]
    for topic, payload in messages:
        publisher.send_multipart([topic.encode(), json.dumps(payload).encode()])

    for topic, sub in subscribers.items():
        expected = [payload for t, payload in messages if t == topic]
        received = []
        for _ in expected:
            frame_topic, body = sub.recv_multipart()
            assert frame_topic.decode() == topic, frame_topic
            received.append(json.loads(body))
        print(f"{topic:8} subscriber received: {[m['headline'] for m in received]}")
        assert received == expected, received
        assert sub.poll(100) == 0, f"{topic} subscriber received a message for another topic"
finally:
    for sub in subscribers.values():
        sub.close()
    publisher.close()
    context.term()

## Example 4: Push-pull pipeline that fans tasks out to three workers

In [ ]:
# Example 4: Push-pull pipeline that fans tasks out to three workers
import zmq
from zmq.utils.monitor import recv_monitor_message

context = zmq.Context()
push = context.socket(zmq.PUSH)
push.setsockopt(zmq.LINGER, 0)
push.setsockopt(zmq.SNDTIMEO, 2000)
port = push.bind_to_random_port("tcp://127.0.0.1")
# A monitor socket reports each completed connection, so the cell can wait for all three workers
monitor = push.get_monitor_socket(zmq.EVENT_HANDSHAKE_SUCCEEDED)
monitor.setsockopt(zmq.RCVTIMEO, 2000)

workers = []
poller = zmq.Poller()
for _ in range(3):
    pull = context.socket(zmq.PULL)
    pull.setsockopt(zmq.LINGER, 0)
    pull.connect(f"tcp://127.0.0.1:{port}")
    workers.append(pull)
    poller.register(pull, zmq.POLLIN)

tasks = [{"task_id": f"task_{i:02d}", "value": i} for i in range(1, 9)]
results = {}
try:
    for _ in workers:
        assert recv_monitor_message(monitor)["event"] == zmq.EVENT_HANDSHAKE_SUCCEEDED
    push.disable_monitor()
    monitor.close()

    for task in tasks:
        push.send_json(task)

    # PUSH spreads the tasks over the connected workers; collect until every task is back or 2 s pass
    while len(results) < len(tasks):
        ready = dict(poller.poll(2000))
        assert ready, f"timed out with {len(results)} of {len(tasks)} tasks received"
        for worker_index, pull in enumerate(workers):
            if pull in ready:
                task = pull.recv_json()
                results[task["task_id"]] = (worker_index, task["value"] ** 2)
finally:
    for pull in workers:
        pull.close()
    push.close()
    context.term()

for task_id, (worker_index, squared) in sorted(results.items()):
    print(f"{task_id}: worker {worker_index} -> {squared}")
assert sorted(results) == [t["task_id"] for t in tasks]
assert [results[t["task_id"]][1] for t in tasks] == [t["value"] ** 2 for t in tasks]
# With all three workers connected, PUSH deals the 8 tasks out round-robin
per_worker = sorted(sum(1 for w, _ in results.values() if w == i) for i in range(len(workers)))
assert per_worker == [2, 3, 3], per_worker

## Example 5: Context and socket options with a PAIR socket exchange

In [ ]:
# Example 5: Context and socket options with a PAIR socket exchange
import json

import zmq

context = zmq.Context()
context.set(zmq.IO_THREADS, 2)
context.set(zmq.MAX_SOCKETS, 1024)
print(f"IO threads: {context.get(zmq.IO_THREADS)}, max sockets: {context.get(zmq.MAX_SOCKETS)}")
assert context.get(zmq.IO_THREADS) == 2
assert context.get(zmq.MAX_SOCKETS) == 1024

left = context.socket(zmq.PAIR)
right = context.socket(zmq.PAIR)
left.setsockopt(zmq.SNDHWM, 100)
left.setsockopt(zmq.RCVHWM, 100)
left.setsockopt(zmq.LINGER, 1000)
assert (left.getsockopt(zmq.SNDHWM), left.getsockopt(zmq.RCVHWM), left.getsockopt(zmq.LINGER)) == (100, 100, 1000)
for sock in (left, right):
    sock.setsockopt(zmq.RCVTIMEO, 2000)

port = left.bind_to_random_port("tcp://127.0.0.1")
right.connect(f"tcp://127.0.0.1:{port}")

messages = [
    {"id": 1, "data": "First message", "size": "small"},
    {"id": 2, "data": "Second message with more content", "size": "medium"},
    {"id": 3, "data": "Third message with even more content to test", "size": "large"},
]
try:
    for msg in messages:
        left.send_string(json.dumps(msg))
        received = json.loads(right.recv_string())
        print(f"Received message {received['id']}: {len(json.dumps(received))} bytes")
        assert received == msg, received
        assert right.getsockopt(zmq.RCVMORE) == 0
    # PAIR is bidirectional
    right.send_string("ack")
    assert left.recv_string() == "ack"
    assert left.getsockopt(zmq.TYPE) == zmq.PAIR
finally:
    left.close(linger=0)
    right.close(linger=0)

assert left.closed and right.closed
context.term()
assert context.closed
print("PAIR exchange completed and resources released")